# Task 2: Filtro de Spam Bayesiano

Objetivo: construir un clasificador Naive Bayes desde cero usando "Bag of Words".

Dataset: archivo entrenamiento.txt donde cada línea es: ETIQUETA \t MENSAJE.

Orden de trabajo:
1. Pre-procesamiento: cargar y limpiar todo el dataset.
2. Separación 80/20: dividir en entrenamiento (80%) y prueba (20%) antes de entrenar.
3. Entrenamiento: calcular probabilidades usando solo el conjunto de entrenamiento.
4. Inferencia: definir la función predict(mensaje).
5. Evaluación: medir desempeño sobre el conjunto de prueba.

## 1. Pre-procesamiento: carga y limpieza del dataset

Cargamos todo el dataset, normalizamos etiquetas y limpiamos el texto (minúsculas, sin puntuación).

In [43]:
# Importar librerías necesarias
import pandas as pd
import math
import random
from collections import Counter
import re

# 1a. Cargar el archivo de datos
data_path = 'entrenamiento.txt'
df = pd.read_csv(
    data_path,
    sep="\t",
    header=None,
    names=["label", "message"],
    encoding="utf-8",
    engine="python",
    quoting=3,
    on_bad_lines='skip'
)

# Normalizar etiquetas a minúsculas y filtrar solo spam/ham
df["label"] = df["label"].astype(str).str.lower()
df = df[df["label"].isin(["spam", "ham"])].copy()
df["message"] = df["message"].astype(str)

print("Paso 1a: datos cargados")
print("Total de mensajes:", df.shape[0])
print("Distribución de clases:")
print(df["label"].value_counts())
df.head()

Paso 1a: datos cargados
Total de mensajes: 5565
Distribución de clases:
label
ham     4818
spam     747
Name: count, dtype: int64


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [44]:
# Separar en conjunto de entrenamiento (80%) y prueba (20%)
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

split_idx = int(0.8 * len(df_shuffled))

train_df = df_shuffled.iloc[:split_idx].copy()
test_df = df_shuffled.iloc[split_idx:].copy()

print("Paso 2: separación 80/20")
print("Tamaño train:", train_df.shape[0])
print("Tamaño test:", test_df.shape[0])
print("Clases en train:")
print(train_df["label"].value_counts())
print("Clases en test:")
print(test_df["label"].value_counts())

Paso 2: separación 80/20
Tamaño train: 4452
Tamaño test: 1113
Clases en train:
label
ham     3857
spam     595
Name: count, dtype: int64
Clases en test:
label
ham     961
spam    152
Name: count, dtype: int64


## 2. Separación 80/20: entrenamiento y prueba

La separación en 80% entrenamiento y 20% prueba se hace antes de entrenar el modelo.

- Train (80%): se usa para calcular vocabulario y probabilidades.
- Test (20%): se usa solo para evaluar el modelo.

In [45]:
# 1b. Función para limpiar texto
def clean_text(s: str) -> str:
    """Limpia el texto:
    - Convierte a minúsculas
    - Elimina puntuación y caracteres especiales
    - Elimina espacios múltiples
    """
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", "", s)  # Solo letras, números y espacios
    s = re.sub(r"\s+", " ", s).strip()  # Limpiar espacios extra
    return s

# Aplicar limpieza a todos los mensajes
df["clean"] = df["message"].apply(clean_text)

print("Paso 1b: texto limpiado (ejemplos)")
df[["label", "message", "clean"]].head(3)

Paso 1b: texto limpiado (ejemplos)


,label,message,clean
0,ham,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...


## 3. Entrenamiento: cálculo de probabilidades

Usamos solo el conjunto de entrenamiento (train_df) para:
1. Generar el vocabulario V.
2. Calcular probabilidades a priori P(spam) y P(ham).
3. Calcular likelihoods con suavizado de Laplace (k = 1).

In [46]:
# 3.1 Generar vocabulario a partir de train_df

# Si por orden de ejecución train_df no tiene la columna 'clean',
# la creamos aquí a partir de la columna 'message'.
if "clean" not in train_df.columns:
    train_df["clean"] = train_df["message"].apply(clean_text)

train_df["tokens"] = train_df["clean"].str.split()

vocab = set(word for toks in train_df["tokens"] for word in toks)
V = len(vocab)

print("Paso 3.1: vocabulario creado")
print("Tamaño del vocabulario:", V)
list(vocab)[:10]

Paso 3.1: vocabulario creado
Tamaño del vocabulario: 8435


['lists',
 'wherres',
 'edwards',
 'wud',
 'legitimat',
 'bec',
 'timeyour',
 '400minscall',
 '3510i',
 'pushbutton']

In [47]:
# 3.4 Construir tabla de likelihoods para cada palabra del vocabulario
likelihood_df = pd.DataFrame(index=sorted(vocab))

# Probabilidades P(palabra | spam) y P(palabra | ham)
likelihood_df["P_w_spam"] = [(spam_counts[w] + k) / den_spam for w in likelihood_df.index]
likelihood_df["P_w_ham"]  = [(ham_counts[w]  + k) / den_ham  for w in likelihood_df.index]

# Usamos logaritmos para evitar underflow numérico
likelihood_df["logP_w_spam"] = likelihood_df["P_w_spam"].apply(math.log)
likelihood_df["logP_w_ham"]  = likelihood_df["P_w_ham"].apply(math.log)

print("Paso 3.4: likelihoods calculados para", len(likelihood_df), "palabras")
likelihood_df.head()

Paso 3.4: likelihoods calculados para 8435 palabras


,P_w_spam,P_w_ham,logP_w_spam,logP_w_ham
0,0.000178,0.000016,-8.633776,-11.042073
008704050406,0.000133,0.000016,-8.921458,-11.042073
0089my,0.000089,0.000016,-9.326923,-11.042073
0121,0.000089,0.000016,-9.326923,-11.042073
01223585334,0.000089,0.000016,-9.326923,-11.042073


In [48]:
# 3.3 Calcular likelihoods con suavizado de Laplace (k = 1)
# Fórmula: P(palabra | clase) = (conteo(palabra en clase) + k) / (total_palabras_clase + k * V)

k = 1.0  # parámetro de suavizado de Laplace

# Extraer todas las palabras por clase
tokens_spam = [w for toks in train_df[train_df["label"] == "spam"]["tokens"] for w in toks]
tokens_ham  = [w for toks in train_df[train_df["label"] == "ham"]["tokens"] for w in toks]

# Contar frecuencias por clase
spam_counts = Counter(tokens_spam)
ham_counts  = Counter(tokens_ham)

# Totales de palabras en cada clase
N_spam = len(tokens_spam)
N_ham  = len(tokens_ham)

# Denominadores con suavizado
den_spam = N_spam + k * V
den_ham  = N_ham  + k * V

print("Paso 3.3: conteos de palabras")
print("Total de palabras en spam:", N_spam)
print("Total de palabras en ham:", N_ham)
print("Denominador spam:", den_spam)
print("Denominador ham:", den_ham)

Paso 3.3: conteos de palabras
Total de palabras en spam: 14038
Total de palabras en ham: 54012
Denominador spam: 22473.0
Denominador ham: 62447.0


In [49]:
# 3.2 Calcular probabilidades a priori (priors)
# P(spam) = proporción de mensajes spam en el conjunto de entrenamiento
# P(ham)  = proporción de mensajes ham en el conjunto de entrenamiento
priors = train_df["label"].value_counts(normalize=True).to_dict()

print("Paso 3.2: priors calculados")
print("P(spam) =", priors.get("spam", 0.0))
print("P(ham)  =", priors.get("ham", 0.0))

Paso 3.2: priors calculados
P(spam) = 0.13364779874213836
P(ham)  = 0.8663522012578616


## 4. Inferencia: función de predicción

Usamos la regla de Bayes para clasificar un nuevo mensaje:

P(spam | mensaje) ∝ P(spam) × producto de P(palabra | spam).

En la implementación usamos logaritmos para sumar en lugar de multiplicar probabilidades.

In [50]:
def predict(message: str) -> str:
    #Clasifica un mensaje como 'spam' o 'ham' usando Naive Bayes.
    
    toks = clean_text(message).split()

    # Comenzar con los log-priors
    score_spam = math.log(priors.get("spam", 1e-10))
    score_ham  = math.log(priors.get("ham", 1e-10))

    # Sumar log-probabilidades de cada palabra del mensaje
    for w in toks:
        if w in vocab:  # solo palabras vistas en entrenamiento
            score_spam += likelihood_df.at[w, "logP_w_spam"]
            score_ham  += likelihood_df.at[w, "logP_w_ham"]

    # Devolver la clase con mayor score
    if score_spam > score_ham:
        return "spam"
    else:
        return "ham"

print("Paso 4: función predict definida")

Paso 4: función predict definida


In [51]:
# Prueba rápida de la función predict

mensajes_prueba = [
    "Congratulations! You won a FREE iPhone! Click here now!!!",
    "Hey mom, can you pick me up after school?",
    "URGENT: Your account will be closed. Call 555-1234!",
    "Thanks for dinner last night, it was great"
]

print("Pruebas de la función predict:\n")
for m in mensajes_prueba:
    print("Mensaje:", m)
    print("Predicción:", predict(m))
    print("---")

Pruebas de la función predict:

Mensaje: Congratulations! You won a FREE iPhone! Click here now!!!
Predicción: spam
---
Mensaje: Hey mom, can you pick me up after school?
Predicción: ham
---
Mensaje: URGENT: Your account will be closed. Call 555-1234!
Predicción: spam
---
Mensaje: Thanks for dinner last night, it was great
Predicción: ham
---


## 5. Evaluación sobre el conjunto de prueba

Usamos el conjunto de prueba (test_df) para medir el desempeño del clasificador.

Calculamos:
- Matriz de confusión.
- Exactitud (accuracy).

In [52]:
# Evaluación en el conjunto de prueba
y_true = test_df["label"].tolist()
y_pred = [predict(m) for m in test_df["message"].tolist()]

# Construir matriz de confusión manualmente
labels = ["spam", "ham"]
idx = {lab: i for i, lab in enumerate(labels)}
cm = [[0, 0], [0, 0]]  # filas = real, columnas = predicho

for t, p in zip(y_true, y_pred):
    cm[idx[t]][idx[p]] += 1

# Extraer valores básicos de la matriz de confusión
tp = cm[0][0]  # spam predicho como spam
fn = cm[0][1]  # spam predicho como ham
fp = cm[1][0]  # ham predicho como spam
tn = cm[1][1]  # ham predicho como ham

print("Matriz de confusión (filas = real, columnas = predicho):")
print("\t\tspam\tham")
print("spam\t\t" + str(tp) + "\t" + str(fn))
print("ham\t\t"  + str(fp) + "\t" + str(tn))

Matriz de confusión (filas = real, columnas = predicho):
		spam	ham
spam		136	16
ham		4	957


In [53]:
# Cálculo de exactitud (accuracy) y métricas básicas

accuracy = (tp + tn) / (tp + tn + fp + fn)

print("Accuracy:", accuracy)

Accuracy: 0.9820305480682839
